# Search engine experiment (decision Q2)

**Question:** should the schedule finder be built on a hand-written *backtracking* search, or on Google's *OR-Tools CP-SAT* solver? See `docs/search-approaches.md`.

This notebook runs both engines on the same data and compares them on five things:

1. **Correctness:** do they find exactly the schedules a brute-force search finds?
2. **Speed to a first schedule** and to a proof that none exists
3. **Many schedules:** speed, order, repeatability
4. **Pinned positions** (an opening act, a closing act, a break)
5. **Your real show** (last section)

Run the cells from the top. The slow ones are the cases where backtracking gives up; the limits in the *Settings* cell keep them short.

This is a *prototype*: once Q2 is decided the winning code moves into `src/performance_scheduling/solver.py` with proper tests, and this notebook stays as the record. Before committing, strip the outputs (`nbstripout`, see `docs/environment.md`) because they can contain real performer names.

## Setup

In [ ]:
from pathlib import Path

import pandas as pd

from performance_scheduling.conflicts import build_conflict_map
from performance_scheduling.io import load_performers

# Find the repo root (the folder holding pyproject.toml), whatever folder this notebook runs from
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
EXAMPLES = ROOT / "data" / "examples"
REAL_FILE = ROOT / "data" / "private" / "2026 S2 performer data-anonymous.csv"   # change if yours is named differently

# ---- Settings: how long to let each engine try before giving up -------------------------
NODE_LIMIT = 300_000      # backtracking: steps before it stops with status "stopped"
TIME_LIMIT = 20.0         # CP-SAT: seconds before it stops with status "stopped"


def setup(path):
    """Load a performer file and return what the search engines need."""
    data = load_performers(path).data
    conflicts = build_conflict_map(data)
    keys = list(data.performance_keys)
    neighbours = {key: set(others) for key, others in conflicts.neighbours.items()}
    # one entry per distinct group of performances a performer is in (duplicates add nothing)
    performer_sets = list({frozenset(v) for v in data.performances_by_performer.values() if len(v) > 1})
    return keys, neighbours, performer_sets

## 1. The two engines

**Backtracking** fills the running order slot by slot from the left, always trying performances in a fixed (alphabetical) order, and undoes a choice when it hits a dead end. A performer who can no longer fit their remaining performances ends a branch early (the "capacity check" from `search-approaches.md`). Its list of schedules is therefore in a fixed, dictionary-like order.

Results have three statuses: **found**, **exhausted** (the whole search finished, so no more exist: a proof) and **stopped** (the limit was reached: proves nothing).

In [ ]:
import time
from itertools import combinations, permutations

BREAK = "BREAK"


class SearchStopped(Exception):
    """Raised inside the search when the node limit is reached."""


class Stats:
    def __init__(self):
        self.nodes = 0


def enumerate_schedules(keys, neighbours, performer_sets, k, breaks, pins=None, order=None,
                        stats=None, node_limit=None):
    """Yield every valid running order, one at a time, in a fixed (lexicographic) order.

    keys           performance keys
    neighbours     key -> set of keys it conflicts with
    performer_sets list of frozensets: the performances each distinct kind of performer is in
    k              fewest slots allowed between two conflicting performances
    breaks         number of breaks to place (pinned breaks count towards this)
    pins           {position (from 1): performance key or BREAK}
    order          the order in which performances are tried (default: alphabetical)
    """
    stats = stats if stats is not None else Stats()
    pins = dict(pins or {})
    order = list(order) if order else sorted(keys)
    n = len(keys)
    total = n + breaks

    pinned_at = {}          # performance -> position
    pinned_breaks = set()
    for position, item in pins.items():
        if not 1 <= position <= total:
            return
        if item == BREAK:
            pinned_breaks.add(position)
        elif item not in neighbours or item in pinned_at:
            return
        else:
            pinned_at[item] = position
    free_breaks = [breaks - len(pinned_breaks)]
    if free_breaks[0] < 0:
        return
    for a, b in combinations(sorted(pinned_at), 2):
        if b in neighbours[a] and abs(pinned_at[a] - pinned_at[b]) < k + 1:
            return

    sets = [frozenset(s) for s in performer_sets]
    sets_of = {key: [] for key in keys}
    for index, s in enumerate(sets):
        for key in s:
            sets_of[key].append(index)
    left = [len(s) for s in sets]
    last = [0] * len(sets)

    seq = []
    placed = set()

    def capacity_ok(position):
        """Can every performer still fit their remaining performances?"""
        for index in range(len(sets)):
            remaining = left[index]
            if remaining:
                earliest = max(position + 1, last[index] + k + 1)
                if earliest + (remaining - 1) * (k + 1) > total:
                    return False
        return True

    def legal(x, position):
        for q in range(max(1, position - k), position):
            item = seq[q - 1]
            if item != BREAK and item in neighbours[x]:
                return False
        for q in range(position + 1, min(total, position + k) + 1):
            item = pins.get(q)
            if item is not None and item != BREAK and item in neighbours[x]:
                return False
        return True

    def search(position):
        stats.nodes += 1
        if node_limit is not None and stats.nodes > node_limit:
            raise SearchStopped
        if position > total:
            yield tuple(seq)
            return
        forced = pins.get(position)
        if forced is not None:
            candidates = [forced]
        else:
            candidates = [x for x in order if x not in placed and x not in pinned_at]
            if free_breaks[0] > 0:
                candidates.append(BREAK)
        for x in candidates:
            if x == BREAK:
                if forced is None:
                    free_breaks[0] -= 1
                seq.append(BREAK)
                if capacity_ok(position):
                    yield from search(position + 1)
                seq.pop()
                if forced is None:
                    free_breaks[0] += 1
            else:
                if not legal(x, position):
                    continue
                saved = [(i, last[i]) for i in sets_of[x]]
                for i in sets_of[x]:
                    left[i] -= 1
                    last[i] = position
                seq.append(x)
                placed.add(x)
                if capacity_ok(position):
                    yield from search(position + 1)
                placed.discard(x)
                seq.pop()
                for i, old in saved:
                    left[i] += 1
                    last[i] = old

    yield from search(1)


def run_backtracking(keys, neighbours, performer_sets, k, breaks, want=1, pins=None, order=None,
                     node_limit=2_000_000):
    """Find up to `want` schedules. Status: found / exhausted (proof) / stopped (limit hit)."""
    stats = Stats()
    found = []
    start = time.perf_counter()
    status = "exhausted"
    try:
        for schedule in enumerate_schedules(keys, neighbours, performer_sets, k, breaks, pins,
                                            order, stats, node_limit):
            found.append(schedule)
            if len(found) >= want:
                status = "found"
                break
        else:
            status = "found" if found else "exhausted"
    except SearchStopped:
        status = "found" if found else "stopped"
    return {"status": status, "schedules": found, "nodes": stats.nodes,
            "seconds": time.perf_counter() - start}

**Brute force** (checks every possible ordering; only for tiny shows) and **CP-SAT** (a constraint solver: we describe the rules and it does the searching).

In [ ]:
def brute_force_valid(keys, neighbours, k, breaks):
    """Every valid schedule, found the slow obvious way (tiny shows only)."""
    total = len(keys) + breaks
    result = set()
    for slots in combinations(range(total), len(keys)):
        for order in permutations(sorted(keys)):
            position = dict(zip(order, slots))
            if all(abs(position[a] - position[b]) >= k + 1
                   for a in keys for b in neighbours[a] if a < b):
                seq = [BREAK] * total
                for key, slot in position.items():
                    seq[slot] = key
                result.add(tuple(seq))
    return result

In [ ]:
def build_cpsat(keys, neighbours, k, breaks, pins=None):
    from ortools.sat.python import cp_model
    total = len(keys) + breaks
    model = cp_model.CpModel()
    position = {x: model.NewIntVar(1, total, x) for x in keys}
    model.AddAllDifferent(list(position.values()))
    for a in keys:
        for b in neighbours[a]:
            if a < b:
                a_first = model.NewBoolVar(f"{a}<{b}")
                model.Add(position[a] + k + 1 <= position[b]).OnlyEnforceIf(a_first)
                model.Add(position[b] + k + 1 <= position[a]).OnlyEnforceIf(a_first.Not())
    for slot, item in (pins or {}).items():
        if item == BREAK:
            for x in keys:
                model.Add(position[x] != slot)
        else:
            model.Add(position[item] == slot)
    return model, position


def to_sequence(values, total):
    seq = [BREAK] * total
    for key, slot in values.items():
        seq[slot - 1] = key
    return tuple(seq)


def run_cpsat(keys, neighbours, k, breaks, want=1, pins=None, time_limit=60.0):
    from ortools.sat.python import cp_model
    total = len(keys) + breaks
    model, position = build_cpsat(keys, neighbours, k, breaks, pins)
    solver = cp_model.CpSolver()
    solver.parameters.max_time_in_seconds = time_limit
    solver.parameters.num_workers = 1
    solver.parameters.random_seed = 1
    start = time.perf_counter()
    found = []
    if want == 1:
        status = solver.Solve(model)
        if status in (cp_model.OPTIMAL, cp_model.FEASIBLE):
            found.append(to_sequence({x: solver.Value(v) for x, v in position.items()}, total))
        outcome = {cp_model.OPTIMAL: "found", cp_model.FEASIBLE: "found",
                   cp_model.INFEASIBLE: "exhausted"}.get(status, "stopped")
    else:
        class Collect(cp_model.CpSolverSolutionCallback):
            def __init__(self):
                super().__init__()
            def on_solution_callback(self):
                found.append(to_sequence({x: self.Value(v) for x, v in position.items()}, total))
                if len(found) >= want:
                    self.StopSearch()
        solver.parameters.enumerate_all_solutions = True
        status = solver.Solve(model, Collect())
        if len(found) >= want:
            outcome = "found"
        elif status in (cp_model.OPTIMAL, cp_model.FEASIBLE):
            outcome = "found" if found else "exhausted"
        elif status == cp_model.INFEASIBLE:
            outcome = "exhausted"
        else:
            outcome = "found" if found else "stopped"
    return {"status": outcome, "schedules": found, "nodes": None,
            "seconds": time.perf_counter() - start}

## 2. Correctness

For each tiny dataset, the full list of schedules from each engine is compared with the brute-force list. **All three must agree exactly**, in count and in content. These are the same counts recorded in `docs/test-datasets.md`.

In [ ]:
rows = []
cases = [("tiny_feasible.csv", [(1, 0), (1, 1), (2, 0), (2, 1), (1, 2)]),
         ("tiny_needs_breaks.csv", [(1, 0), (1, 1), (1, 2)]),
         ("tiny_no_conflicts.csv", [(1, 0)])]
for name, settings in cases:
    keys, nb, sets = setup(EXAMPLES / name)
    for k, breaks in settings:
        truth = brute_force_valid(keys, nb, k, breaks)
        by_backtracking = list(enumerate_schedules(keys, nb, sets, k, breaks))
        by_cpsat = run_cpsat(keys, nb, k, breaks, want=10**6, time_limit=60)["schedules"]
        rows.append({"dataset": name, "gap k": k, "breaks": breaks, "brute force": len(truth),
                     "backtracking": len(by_backtracking), "cp-sat": len(by_cpsat),
                     "backtracking matches": set(by_backtracking) == truth and len(by_backtracking) == len(truth),
                     "cp-sat matches": set(by_cpsat) == truth and len(by_cpsat) == len(truth)})
pd.DataFrame(rows)

## 3. Speed on realistic shows (about 20 performances)

Each row asks for **one schedule**. `found` means a schedule was produced; `exhausted` means the engine *proved* none exists; `stopped` means it gave up at the limit.

Also shown first: a quick bound that needs no search. If *m* performances all clash with each other (a "clique"), they need at least `(m - 1)(k + 1) + 1` slots. That is a cheap way to prove some shows impossible, and a good explanation to give an organiser.

In [ ]:
def max_clique(keys, neighbours):
    """The largest group of performances that all clash with each other."""
    best = []

    def expand(chosen, candidates, excluded):
        nonlocal best
        if not candidates and not excluded:
            if len(chosen) > len(best):
                best = list(chosen)
            return
        if len(chosen) + len(candidates) <= len(best):
            return
        pivot = max(candidates | excluded, key=lambda v: len(candidates & neighbours[v]))
        for v in list(candidates - neighbours[pivot]):
            expand(chosen + [v], candidates & neighbours[v], excluded & neighbours[v])
            candidates = candidates - {v}
            excluded = excluded | {v}

    expand([], set(keys), set())
    return best


def slots_needed(group_size, k):
    return (group_size - 1) * (k + 1) + 1


for name in ["typical_20_performances.csv", "one_busy_performer.csv", "dense_overlap.csv", "impossible_dense.csv"]:
    keys, nb, sets = setup(EXAMPLES / name)
    clique = max_clique(keys, nb)
    print(f"{name:<30} {len(keys)} performances, largest group that all clash: {len(clique)}; "
          f"needs {slots_needed(len(clique), 1)} slots at k=1, {slots_needed(len(clique), 2)} at k=2, "
          f"{slots_needed(len(clique), 3)} at k=3")

In [ ]:
def one_row(label, keys, nb, sets, k, breaks, pins=None):
    a = run_backtracking(keys, nb, sets, k, breaks, want=1, pins=pins, node_limit=NODE_LIMIT)
    b = run_cpsat(keys, nb, k, breaks, want=1, pins=pins, time_limit=TIME_LIMIT)
    return {"dataset": label, "k": k, "breaks": breaks,
            "backtracking": a["status"], "bt seconds": round(a["seconds"], 2), "bt steps": a["nodes"],
            "cp-sat": b["status"], "cp-sat seconds": round(b["seconds"], 2)}


plan = [("typical_20_performances.csv", [(1, 0), (2, 0), (2, 4), (3, 0), (3, 4)]),
        ("one_busy_performer.csv", [(1, 0), (1, 2), (2, 0), (2, 5)]),
        ("dense_overlap.csv", [(1, 0), (2, 0), (2, 4)]),
        ("impossible_dense.csv", [(1, 0), (1, 2), (2, 0)])]
rows = []
for name, settings in plan:
    keys, nb, sets = setup(EXAMPLES / name)
    for k, breaks in settings:
        rows.append(one_row(name, keys, nb, sets, k, breaks))
pd.DataFrame(rows)

## 4. Many schedules: speed, order and repeatability

The finder must hand out schedules on demand, in a repeatable order, without repeats (decisions D4 and D20).

In [ ]:
keys, nb, sets = setup(EXAMPLES / "typical_20_performances.csv")
rows = []
for want in (1, 100, 1000):
    a = run_backtracking(keys, nb, sets, 1, 0, want=want, node_limit=5_000_000)
    b = run_cpsat(keys, nb, 1, 0, want=want, time_limit=60)
    rows.append({"schedules wanted": want,
                 "backtracking found": len(a["schedules"]), "bt seconds": round(a["seconds"], 2),
                 "cp-sat found": len(b["schedules"]), "cp-sat seconds": round(b["seconds"], 2)})
display(pd.DataFrame(rows))

bt = run_backtracking(keys, nb, sets, 1, 0, want=100)["schedules"]
cp1 = run_cpsat(keys, nb, 1, 0, want=100)["schedules"]
cp2 = run_cpsat(keys, nb, 1, 0, want=100)["schedules"]


def slots_differing(x, y):
    return sum(1 for p, q in zip(x, y) if p != q)


print("Same list when run twice?   backtracking: True (no randomness)   cp-sat:", cp1 == cp2)
print("In dictionary order?        backtracking:", bt == sorted(bt), "  cp-sat:", cp1 == sorted(cp1))
print("Slots that differ between neighbouring schedules (average):  backtracking:",
      round(sum(slots_differing(bt[i], bt[i + 1]) for i in range(99)) / 99, 1),
      "  cp-sat:", round(sum(slots_differing(cp1[i], cp1[i + 1]) for i in range(99)) / 99, 1))
print("Schedules in common between the two lists of 100:", len(set(bt) & set(cp1)))

## 5. Pinned positions

An organiser fixes an opening act, a closing act and a break. Two checks: easy pins, then hard ones (the two performances with the most clashes pinned to the ends, at a larger gap). Finally, the pins that **contradict** each other, and which pins CP-SAT names as the cause.

In [ ]:
keys, nb, sets = setup(EXAMPLES / "typical_20_performances.csv")
total = len(keys)
rows = []

easy = {1: "bollywood (open)", total + 2: "kpop girls (intermediate)", 11: BREAK}
rows.append(one_row("easy pins (k=1, 2 breaks)", keys, nb, sets, 1, 2, pins=easy))

ranked = sorted(keys, key=lambda x: -len(nb[x]))
for k, breaks in [(1, 0), (2, 4)]:
    hard = {1: ranked[0], total + breaks: ranked[1]}
    rows.append(one_row(f"hard pins: most-clashing first/last (k={k})", keys, nb, sets, k, breaks, pins=hard))

clashing = sorted(nb["bollywood (open)"])[0]
rows.append(one_row("contradicting pins (clashing acts side by side)", keys, nb, sets, 1, 2,
                    pins={1: "bollywood (open)", 2: clashing}))
pd.DataFrame(rows)

In [ ]:
from ortools.sat.python import cp_model


def blocking_pins(keys, nb, k, breaks, pins):
    """Ask CP-SAT which pins cannot all hold together (an 'unsat core' of assumptions)."""
    total = len(keys) + breaks
    model = cp_model.CpModel()
    position = {x: model.NewIntVar(1, total, x) for x in keys}
    model.AddAllDifferent(list(position.values()))
    for a in keys:
        for b in nb[a]:
            if a < b:
                a_first = model.NewBoolVar("")
                model.Add(position[a] + k + 1 <= position[b]).OnlyEnforceIf(a_first)
                model.Add(position[b] + k + 1 <= position[a]).OnlyEnforceIf(a_first.Not())
    labels = {}
    for slot, item in pins.items():
        lit = model.NewBoolVar(f"pin{slot}")
        labels[lit.Index()] = (slot, item)
        if item == BREAK:
            for x in keys:
                model.Add(position[x] != slot).OnlyEnforceIf(lit)
        else:
            model.Add(position[item] == slot).OnlyEnforceIf(lit)
        model.AddAssumption(lit)
    solver = cp_model.CpSolver()
    solver.parameters.num_workers = 1
    if solver.Solve(model) != cp_model.INFEASIBLE:
        return None
    return [labels[i] for i in solver.SufficientAssumptionsForInfeasibility()]


# four pins; two of them (slots 1 and 2) clash with each other, the other two are innocent
pins = {1: "bollywood (open)", 2: clashing, 10: BREAK, total + 2: "kpop girls (intermediate)"}
print("Pins that cannot all hold together:", blocking_pins(keys, nb, 1, 2, pins))

## 6. Your real show

Set `REAL_FILE` in the *Setup* cell if needed. The gaps and break counts below are the ones worth trying on a 19-performance show; change them freely. The results of this section decide how large a gap your real shows can support and how comfortable each engine is on them.

(If the file is not found, this section just prints a message.)

In [ ]:
if not REAL_FILE.exists():
    print("Real file not found:", REAL_FILE)
else:
    keys, nb, sets = setup(REAL_FILE)
    clique = max_clique(keys, nb)
    print(f"{len(keys)} performances; largest group that all clash: {len(clique)}")
    for k in (1, 2, 3, 4):
        print(f"  k={k}: that group alone needs {slots_needed(len(clique), k)} slots "
              f"(show has {len(keys)} + breaks)")
    rows = []
    for k, breaks in [(1, 0), (2, 0), (2, 2), (3, 0), (3, 2), (3, 4)]:
        rows.append(one_row("real show", keys, nb, sets, k, breaks))
    display(pd.DataFrame(rows))

In [ ]:
if REAL_FILE.exists():
    rows = []
    for want in (1, 100):
        a = run_backtracking(keys, nb, sets, 1, 0, want=want, node_limit=5_000_000)
        b = run_cpsat(keys, nb, 1, 0, want=want, time_limit=60)
        rows.append({"schedules wanted": want, "backtracking found": len(a["schedules"]),
                     "bt seconds": round(a["seconds"], 2), "cp-sat found": len(b["schedules"]),
                     "cp-sat seconds": round(b["seconds"], 2)})
    display(pd.DataFrame(rows))

## 7. Notes

Write what you see here before we decide Q2:

- Which engine found schedules fastest on your real show?
- Did either engine give up (`stopped`) on anything you would want an answer for?
- What is the largest gap `k` your show supports, and with how many breaks?